# MSE & GMS Budget — Exploration

Investigates the three budget methods, diagnoses the boundary-term problem, and implements the unified mass correction fix.

**Script:** `scripts/mse_budget.py`  
**Notes:** `notes/MSE_budget_methods.md`

| Section | Topic |
|---|---|
| 0 | Setup |
| 1 | Baseline — all three methods (raw) |
| 2 | Omega profiles — why shape matters |
| 3 | The h profile and campaign mean |
| 4 | The boundary term problem (diagnosed) |
| 5 | Idea A — explicit boundary correction |
| 6 | **Unified mass correction — the best fix** ⭐ |
| 7 | Idea C — anomaly MSE (alternative) |
| 8 | GMS comparison across all approaches |

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from scripts.mse_budget import (
    load_dataset,
    compute_budget,
    apply_mass_correction,
    compute_boundary_term,
    omega_mass_corrected,
    method1_mass_corrected,
    anomaly_mse,
    method2_flux_anomaly,
    _mse, _dse, G,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
TOP_COLOR = '#d62728'
BOT_COLOR = '#1f77b4'

In [ ]:
ds     = load_dataset()
alt    = ds['altitude'].values
p_m    = ds['p_mean'].values      # (circle, altitude) Pa
cat    = ds['category_evolutionary'].values
angle  = ds['top_heaviness_angle'].values
omega  = ds['omega'].values       # (circle, altitude) Pa/s

mask_top = np.array(['Top-Heavy'    in str(c) for c in cat])
mask_bot = np.array(['Bottom-Heavy' in str(c) for c in cat])

print(f'Circles: {ds.sizes["circle"]}  |  Top-Heavy: {mask_top.sum()}  |  Bottom-Heavy: {mask_bot.sum()}')

## 1. Baseline — All Three Methods

In [ ]:
budget = compute_budget(ds)

for label, mask in [('Top-Heavy', mask_top), ('Bottom-Heavy', mask_bot)]:
    va  = budget['vert_adv'].values[mask]
    vds = budget['vert_adv_dse'].values[mask]
    ha  = budget['horiz_adv'].values[mask]
    fd  = budget['flux_div_mse'].values[mask]
    bnd = budget['h_div_residual'].values[mask]
    g1  = budget['gms_adv'].values[mask]

    gms_group = np.nanmean(va) / np.nanmean(vds) if abs(np.nanmean(vds)) > 0 else np.nan

    print(f'\n── {label} (n={mask.sum()}) ──')
    print(f'  Method 1  <ω ∂h/∂p>   = {np.nanmean(va):+8.1f} W/m²')
    print(f'            <v·∇h>      = {np.nanmean(ha):+8.1f} W/m²')
    print(f'            GMS_adv     = {gms_group:+.4f}  (median per-circle = {np.nanmedian(g1):+.4f})')
    print(f'  Method 2  <∇·(vh)>    = {np.nanmean(fd):+8.1f} W/m²  ← large = boundary term')
    print(f'  Method 3  boundary    = {np.nanmean(bnd):+8.1f} W/m²  ← dominates residual')

## 2. Omega Profiles — Why Shape Matters for GMS

In [ ]:
om_top = np.nanmean(omega[mask_top], axis=0)
om_bot = np.nanmean(omega[mask_bot], axis=0)
p_top  = np.nanmean(p_m[mask_top], axis=0) / 100   # hPa for display
p_bot  = np.nanmean(p_m[mask_bot], axis=0) / 100
ok     = p_top > 200   # troposphere only

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

for ax, om, pm, color, label in [
    (axes[0], om_top, p_top, TOP_COLOR, 'Top-Heavy'),
    (axes[1], om_bot, p_bot, BOT_COLOR, 'Bottom-Heavy'),
]:
    ax.plot(om[ok], pm[ok], color=color, lw=2.5)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.invert_yaxis()
    ax.set_xlabel(r'$\omega$ (Pa s$^{-1}$)')
    ax.set_ylabel('Pressure (hPa)')
    ax.set_title(f'{label} — group mean $\\omega$(p)')
    ax.set_ylim(1000, 200)
    ax.grid(True, alpha=0.3)

fig.suptitle('Group-mean omega profiles', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. The h Profile and Campaign Mean

$h \approx 3.5 \times 10^5$ J kg$^{-1}$ is enormous compared to circle-to-circle anomalies.  
This motivates subtracting the campaign mean (Idea C) to remove the large background.

In [ ]:
h_prime, s_prime, h_mean, s_mean = anomaly_mse(ds)

h_prof = _mse(ds['ta_mean'].values, alt[np.newaxis, :], ds['q_mean'].values)
s_prof = _dse(ds['ta_mean'].values, alt[np.newaxis, :])

p_disp = np.nanmean(p_m, axis=0) / 100
ok     = p_disp > 200

h_prime_top = np.nanmean(h_prime[mask_top], axis=0)
h_prime_bot = np.nanmean(h_prime[mask_bot], axis=0)
h_std       = np.nanstd(h_prof,  axis=0)
hp_std      = np.nanstd(h_prime, axis=0)

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

axes[0].plot(h_mean[ok] / 1e3, p_disp[ok], 'k-',  lw=2, label='h (MSE)')
axes[0].plot(s_mean[ok] / 1e3, p_disp[ok], 'k--', lw=2, label='s (DSE)')
axes[0].set_xlabel('Energy (kJ kg$^{-1}$)')
axes[0].set_title('Campaign-mean h and s')
axes[0].legend()

axes[1].plot(h_prime_top[ok] / 1e3, p_disp[ok], color=TOP_COLOR, lw=2, label="Top-Heavy h'")
axes[1].plot(h_prime_bot[ok] / 1e3, p_disp[ok], color=BOT_COLOR, lw=2, label="Bottom-Heavy h'")
axes[1].axvline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel("h' = h − h̄ (kJ kg$^{-1}$)")
axes[1].set_title("Group-mean MSE anomaly h'")
axes[1].legend()

axes[2].plot(h_std[ok]  / 1e3, p_disp[ok], 'k-',  lw=2, label='std(h)')
axes[2].plot(hp_std[ok] / 1e3, p_disp[ok], 'r--', lw=2, label="std(h')")
axes[2].set_xlabel('Std across circles (kJ kg$^{-1}$)')
axes[2].set_title("Variability: h vs h'")
axes[2].legend()

for ax in axes:
    ax.invert_yaxis()
    ax.set_ylim(1000, 200)
    ax.set_ylabel('Pressure (hPa)')
    ax.grid(True, alpha=0.3)

fig.suptitle('MSE profiles: full vs anomaly', fontweight='bold')
plt.tight_layout()
plt.show()

mid = len(h_std[ok]) // 2
print(f'At ~500 hPa:')
print(f'  Campaign-mean h = {h_mean[ok][mid]/1e3:.1f} kJ/kg')
print(f'  std(h)          = {h_std[ok][mid]/1e3:.3f} kJ/kg')
print(f"  std(h')         = {hp_std[ok][mid]/1e3:.3f} kJ/kg")
print(f"  → full h is ~{h_mean[ok][mid]/hp_std[ok][mid]:.0f}× larger than circle-to-circle anomaly")

## 4. The Boundary Term Problem

The identity $\langle h \cdot \nabla \cdot \mathbf{v} \rangle = \langle \omega \, \partial h / \partial p \rangle$ holds only when $\omega = 0$ at both boundaries.  
BEACH profiles stop at ~16 km; the tropopause is ~17 km — so $\omega_{\rm top} \neq 0$.

In [ ]:
bnd_ds   = compute_boundary_term(ds)
bnd_vals = bnd_ds['boundary_term'].values
om_top_v = bnd_ds['omega_top'].values
va_vals  = budget['vert_adv'].values
g1_vals  = budget['gms_adv'].values

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# omega at profile top per circle
axes[0].scatter(om_top_v[mask_top], np.arange(mask_top.sum()),
                color=TOP_COLOR, s=30, label='Top-Heavy')
axes[0].scatter(om_top_v[mask_bot], np.arange(mask_bot.sum()),
                color=BOT_COLOR, s=30, marker='s', label='Bottom-Heavy')
axes[0].axvline(0, color='k', lw=0.8)
axes[0].set_xlabel(r'$\omega_{\rm top}$ (Pa s$^{-1}$)')
axes[0].set_title(r'$\omega$ at highest valid level')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# boundary term distribution
axes[1].hist(bnd_vals[mask_top], bins=15, color=TOP_COLOR, alpha=0.6, label='Top-Heavy')
axes[1].hist(bnd_vals[mask_bot], bins=15, color=BOT_COLOR, alpha=0.6, label='Bottom-Heavy')
axes[1].axvline(0, color='k', lw=0.8)
axes[1].set_xlabel(r'$h_{\rm top}\,\omega_{\rm top}\,/\,g$ (W m$^{-2}$)')
axes[1].set_title('Boundary term distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# boundary term vs vertical advection
axes[2].scatter(va_vals[mask_top], bnd_vals[mask_top], color=TOP_COLOR, s=40, label='Top-Heavy')
axes[2].scatter(va_vals[mask_bot], bnd_vals[mask_bot], color=BOT_COLOR, s=40, marker='s', label='Bottom-Heavy')
lim = np.nanpercentile(np.abs(np.concatenate([va_vals, bnd_vals])), 95) * 1.2
axes[2].plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, label='1:1')
axes[2].axhline(0, color='k', lw=0.5)
axes[2].axvline(0, color='k', lw=0.5)
axes[2].set_xlabel(r'$\langle\omega\,\partial h/\partial p\rangle$ (W m$^{-2}$)')
axes[2].set_ylabel(r'$h_{\rm top}\omega_{\rm top}/g$ (W m$^{-2}$)')
axes[2].set_title('Boundary term vs vert. advection')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

fig.suptitle('Section 4 — Boundary term quantified', fontweight='bold')
plt.tight_layout()
plt.show()

for label, mask in [('Top-Heavy', mask_top), ('Bottom-Heavy', mask_bot)]:
    print(f'{label}: mean boundary term = {np.nanmean(bnd_vals[mask]):+.1f} W/m²'
          f'   mean <ω∂h/∂p> = {np.nanmean(va_vals[mask]):+.1f} W/m²')

## 5. Idea A — Explicit Boundary Correction

Subtract $h_{\rm top}\,\omega_{\rm top}\,/\,g$ from Method 2's flux divergence:
$$\langle \nabla \cdot (\mathbf{v}h) \rangle_{\rm corr} = \langle \nabla \cdot (\mathbf{v}h) \rangle_{\rm M2} - \frac{h_{\rm top}\,\omega_{\rm top}}{g}$$

**Limitation:** requires knowing exactly which end BEACH uses as its $\omega=0$ boundary condition.

In [ ]:
fd_raw  = budget['flux_div_mse'].values
fds_raw = budget['flux_div_dse'].values

# DSE boundary term
s_prof_all = _dse(ds['ta_mean'].values, alt[np.newaxis, :])
s_top      = np.full(ds.sizes['circle'], np.nan)
for i in range(ds.sizes['circle']):
    valid = np.isfinite(omega[i]) & np.isfinite(s_prof_all[i])
    if valid.sum() > 2:
        s_top[i] = s_prof_all[i, np.where(valid)[0][-1]]
bnd_dse = s_top * bnd_ds['omega_top'].values / G

fd_A   = fd_raw  - bnd_vals
fds_A  = fds_raw - bnd_dse
gms_A  = np.where(np.abs(fds_A) > 10, fd_A / fds_A, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(va_vals[mask_top], fd_A[mask_top], color=TOP_COLOR, s=50, label='Top-Heavy')
axes[0].scatter(va_vals[mask_bot], fd_A[mask_bot], color=BOT_COLOR, s=50, marker='s', label='Bottom-Heavy')
lim = np.nanpercentile(np.abs(np.concatenate([va_vals, fd_A])), 95) * 1.3
axes[0].plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, label='1:1')
axes[0].axhline(0, color='k', lw=0.5); axes[0].axvline(0, color='k', lw=0.5)
axes[0].set_xlabel(r'Method 1 $\langle\omega\partial h/\partial p\rangle$ (W m$^{-2}$)')
axes[0].set_ylabel(r'Method 2 corrected (W m$^{-2}$)')
axes[0].set_title('Idea A: corrected flux vs vert. advection')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].scatter(g1_vals[mask_top], gms_A[mask_top], color=TOP_COLOR, s=50, label='Top-Heavy')
axes[1].scatter(g1_vals[mask_bot], gms_A[mask_bot], color=BOT_COLOR, s=50, marker='s', label='Bottom-Heavy')
lim2 = 3
axes[1].plot([-lim2, lim2], [-lim2, lim2], 'k--', lw=0.8, label='1:1')
axes[1].set_xlim(-lim2, lim2); axes[1].set_ylim(-lim2, lim2)
axes[1].set_xlabel('GMS_adv (Method 1)')
axes[1].set_ylabel('GMS corrected (Idea A)')
axes[1].set_title('GMS: Method 1 vs Idea A')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.suptitle('Idea A — Explicit boundary-term subtraction', fontweight='bold')
plt.tight_layout()
plt.show()

for label, mask in [('Top-Heavy', mask_top), ('Bottom-Heavy', mask_bot)]:
    print(f'{label}: flux corrected = {np.nanmean(fd_A[mask]):+.1f} W/m²'
          f'   GMS_A median = {np.nanmedian(gms_A[mask]):.3f}')

## 6. Unified Mass Correction — The Best Fix

**Strategy:** `compute_budget(ds, mass_correct=True)` calls `apply_mass_correction(ds)` first, which:
1. Applies a **linear-in-pressure ramp** to omega, forcing ω = 0 at the top of the integration domain
2. Subtracts the corresponding **depth-uniform Δdiv** from the BEACH div field (continuity: ∂ω/∂p = −div)
3. Re-runs all three methods on the corrected dataset

**Key insight — why the valid mask matters:**  
`omega_mass_corrected` must use the same valid-point mask as `_vadv_col` and `_col_int_p` (requiring all of omega, p, *and* h to be finite). Using a looser mask (omega & p only) zeros omega at a level above the integration domain — the boundary term is unaffected. With the correct mask, the ramp zeros omega *exactly* at the top of the integration domain.

**What happens to the results:**  

| Quantity | Before | After | Notes |
|---|---|---|---|
| Boundary term \|mean\| | ~3877 W/m² | ~22 W/m² | 99.4% reduction |
| Residual 22 W/m² | — | irreducible | div–ω inconsistency in BEACH |
| GMS_adv (group, Top-Heavy) | +0.29 | +0.32 | stable ✅ |
| GMS_flux (group, Top-Heavy) | unusable | +0.28 | now valid for comparison |
| Δdiv | — | ~10⁻⁷ s⁻¹ | tiny — confirms BEACH nearly mass-conserved |

**Note on GMS interpretation:**  
Use **ratio of group means** (e.g. `mean(vert_adv)/mean(vert_adv_dse)`) rather than `mean(per-circle GMS)`.  
The per-circle ratio is noisy when vert_adv_dse is near zero; the group-level ratio is the physically meaningful quantity.

In [ ]:
# ── Run unified mass-corrected budget ─────────────────────────────────────
budget_mc = compute_budget(ds, mass_correct=True)

va_mc   = budget_mc['vert_adv'].values
vds_mc  = budget_mc['vert_adv_dse'].values
fd_mc   = budget_mc['flux_div_mse'].values
fds_mc  = budget_mc['flux_div_dse'].values
bnd_mc  = budget_mc['h_div_residual'].values   # should be near 0
g1_mc   = budget_mc['gms_adv'].values
g2_mc   = budget_mc['gms_flux'].values
dd_mc   = budget_mc['delta_div'].values

bnd_raw = budget['h_div_residual'].values

# ── Panel 1: closure check ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for color, mask, label in [(TOP_COLOR, mask_top, 'Top-Heavy'),
                            (BOT_COLOR, mask_bot, 'Bottom-Heavy')]:
    axes[0].hist(bnd_raw[mask], bins=12, color=color, alpha=0.4, label=f'{label} raw')
    axes[0].hist(bnd_mc[mask],  bins=12, color=color, alpha=0.9,
                 histtype='step', lw=2.5, label=f'{label} corrected')

axes[0].axvline(0, color='k', lw=1)
axes[0].set_xlabel(r'$\langle h\nabla\cdot\mathbf{v}\rangle - \langle\omega\partial h/\partial p\rangle$ (W m$^{-2}$)')
axes[0].set_title('Closure check: boundary term before / after')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# ── Panel 2: M1 vs M2 vertical term after correction ──────────────────────
# After correction h_div_col ≈ vert_adv (boundary term → 0)
axes[1].scatter(va_mc[mask_top], bnd_mc[mask_top] + va_mc[mask_top],
                color=TOP_COLOR, s=50, label='Top-Heavy')
axes[1].scatter(va_mc[mask_bot], bnd_mc[mask_bot] + va_mc[mask_bot],
                color=BOT_COLOR, s=50, marker='s', label='Bottom-Heavy')
lim = np.nanpercentile(np.abs(va_mc[np.isfinite(va_mc)]), 95) * 1.3
axes[1].plot([-lim, lim], [-lim, lim], 'k--', lw=0.9, label='1:1')
axes[1].axhline(0, color='k', lw=0.5); axes[1].axvline(0, color='k', lw=0.5)
axes[1].set_xlabel(r'$\langle\omega\partial h/\partial p\rangle$ Method 1 (W m$^{-2}$)')
axes[1].set_ylabel(r'$\langle h\nabla\cdot\mathbf{v}\rangle$ Method 2 (W m$^{-2}$)')
axes[1].set_title(r'$\langle h\cdot\nabla\cdot v\rangle \approx \langle\omega\partial h/\partial p\rangle$ after fix')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# ── Panel 3: Δdiv distribution (should be small) ──────────────────────────
axes[2].hist(dd_mc[mask_top] * 1e6, bins=12, color=TOP_COLOR, alpha=0.6, label='Top-Heavy')
axes[2].hist(dd_mc[mask_bot] * 1e6, bins=12, color=BOT_COLOR, alpha=0.6, label='Bottom-Heavy')
axes[2].axvline(0, color='k', lw=0.8)
axes[2].set_xlabel(r'$\Delta$div (×10$^{-6}$ s$^{-1}$)')
axes[2].set_title('Magnitude of mass correction per circle')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

fig.suptitle('Section 6 — Unified mass correction (linear ramp + consistent div)',
             fontweight='bold')
plt.tight_layout()
plt.show()

# ── Group-level GMS comparison (ratio of means — more stable than mean of ratios)
print(f'\n{"Quantity":<38} {"Top-Heavy":>12} {"Bottom-Heavy":>13}')
print('─' * 65)
print(f'{"Boundary term raw |mean| [W/m²]":<38} '
      f'{np.nanmean(np.abs(bnd_raw[mask_top])):>12.1f} '
      f'{np.nanmean(np.abs(bnd_raw[mask_bot])):>13.1f}')
print(f'{"Boundary term mc  |mean| [W/m²]":<38} '
      f'{np.nanmean(np.abs(bnd_mc[mask_top])):>12.2f} '
      f'{np.nanmean(np.abs(bnd_mc[mask_bot])):>13.2f}')
print(f'{"Reduction":<38} '
      f'{np.nanmean(np.abs(bnd_raw[mask_top]))/np.nanmean(np.abs(bnd_mc[mask_top])):>11.0f}× '
      f'{np.nanmean(np.abs(bnd_raw[mask_bot]))/np.nanmean(np.abs(bnd_mc[mask_bot])):>12.0f}×')

for lab, m in [('Top-Heavy', mask_top), ('Bottom-Heavy', mask_bot)]:
    va_r  = np.nanmean(budget['vert_adv'].values[m])
    vds_r = np.nanmean(budget['vert_adv_dse'].values[m])
    va_m  = np.nanmean(va_mc[m]);   vds_m = np.nanmean(vds_mc[m])
    fd_m  = np.nanmean(fd_mc[m]);   fds_m = np.nanmean(fds_mc[m])
    print(f'\n{lab}:')
    print(f'  GMS_adv raw  (ratio of group means) = {va_r/vds_r:+.4f}')
    print(f'  GMS_adv mc   (ratio of group means) = {va_m/vds_m:+.4f}  ← stable')
    print(f'  GMS_flux mc  (ratio of group means) = {fd_m/fds_m:+.4f}  ← includes horiz. advection')
    rel = np.nanmean(np.abs(bnd_mc[m])) / np.nanmean(np.abs(va_mc[m])) * 100
    print(f'  Residual closure error = {rel:.1f}%  (irreducible div–ω inconsistency in BEACH)')

## 6b. Omega Profiles — Raw vs Mass-Corrected

Compare group-mean omega profiles before and after the O'Brien linear ramp.
The correction is physically small (~1–2% of div), so shapes are nearly identical.
Any visible shift at the profile top confirms the ramp is acting at the correct level.

In [ ]:
# ── Omega replot: raw vs mass-corrected ──────────────────────────────────
omega_mc, _ = omega_mass_corrected(ds)
p_disp      = np.nanmean(p_m, axis=0) / 100   # hPa for display only

om_top_raw = np.nanmean(omega[mask_top],    axis=0)
om_bot_raw = np.nanmean(omega[mask_bot],    axis=0)
om_top_mc  = np.nanmean(omega_mc[mask_top], axis=0)
om_bot_mc  = np.nanmean(omega_mc[mask_bot], axis=0)
p_top_hpa  = np.nanmean(p_m[mask_top], axis=0) / 100
p_bot_hpa  = np.nanmean(p_m[mask_bot], axis=0) / 100
ok         = p_disp > 200   # troposphere

fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)

# ── Left: Top-Heavy raw vs corrected ──────────────────────────────────────
axes[0].plot(om_top_raw[ok], p_top_hpa[ok], color=TOP_COLOR, lw=2.5, label='raw')
axes[0].plot(om_top_mc[ok],  p_top_hpa[ok], color=TOP_COLOR, lw=2.5, ls='--', label='mass-corrected')
axes[0].axvline(0, color='k', lw=0.8, ls=':')
axes[0].invert_yaxis(); axes[0].set_ylim(1000, 200)
axes[0].set_xlabel(r'$\omega$ (Pa s$^{-1}$)'); axes[0].set_ylabel('Pressure (hPa)')
axes[0].set_title('Top-Heavy group mean $\omega$')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# ── Centre: Bottom-Heavy raw vs corrected ─────────────────────────────────
axes[1].plot(om_bot_raw[ok], p_bot_hpa[ok], color=BOT_COLOR, lw=2.5, label='raw')
axes[1].plot(om_bot_mc[ok],  p_bot_hpa[ok], color=BOT_COLOR, lw=2.5, ls='--', label='mass-corrected')
axes[1].axvline(0, color='k', lw=0.8, ls=':')
axes[1].invert_yaxis(); axes[1].set_ylim(1000, 200)
axes[1].set_xlabel(r'$\omega$ (Pa s$^{-1}$)')
axes[1].set_title('Bottom-Heavy group mean $\omega$')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# ── Right: difference (raw − corrected) ──────────────────────────────────
diff_top = om_top_raw - om_top_mc
diff_bot = om_bot_raw - om_bot_mc
axes[2].plot(diff_top[ok], p_top_hpa[ok], color=TOP_COLOR, lw=2.5, label='Top-Heavy diff')
axes[2].plot(diff_bot[ok], p_bot_hpa[ok], color=BOT_COLOR, lw=2.5, label='Bottom-Heavy diff')
axes[2].axvline(0, color='k', lw=0.8, ls=':')
axes[2].invert_yaxis(); axes[2].set_ylim(1000, 200)
axes[2].set_xlabel(r'$\Delta\omega$ raw − corrected (Pa s$^{-1}$)')
axes[2].set_title('Correction magnitude per level')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

fig.suptitle('Section 6b — Omega profiles: raw vs mass-corrected', fontweight='bold')
plt.tight_layout()
plt.show()

# Summary stats
for label, mask, om_r, om_c in [
    ('Top-Heavy',    mask_top, om_top_raw, om_top_mc),
    ('Bottom-Heavy', mask_bot, om_bot_raw, om_bot_mc),
]:
    max_diff = np.nanmax(np.abs(om_r[ok] - om_c[ok]))
    rms_raw  = np.sqrt(np.nanmean(om_r[ok]**2))
    print(f"{label}: max |Δω| = {max_diff:.2e} Pa/s  |  RMS(ω) = {rms_raw:.2e} Pa/s  →  correction is {max_diff/rms_raw*100:.1f}% of signal")

## 7. Idea C — Anomaly MSE *(most promising)*

Replace $h \to h' = h - \bar{h}(z)$ in the mass-divergence term of Method 2.  
The campaign mean $\bar{h}(z)$ has no horizontal gradient, so $\partial h'/\partial x = \partial h/\partial x$ — horizontal advection is unchanged.  
The boundary error $\propto h'_{\rm top}$ drops from $\sim 3.5 \times 10^5$ to $\sim 10^3$ J kg$^{-1}$.

In [ ]:
fa_ds = method2_flux_anomaly(ds)

fd_C   = fa_ds['flux_div_mse_a'].values
fds_C  = fa_ds['flux_div_dse_a'].values
gms_C  = fa_ds['gms_flux_a'].values

# Reduced boundary term with h'
bnd_anomaly = np.full(ds.sizes['circle'], np.nan)
for i in range(ds.sizes['circle']):
    valid = np.isfinite(omega[i]) & np.isfinite(h_prime[i])
    if valid.sum() > 2:
        idx_top_i = np.where(valid)[0][-1]
        bnd_anomaly[i] = h_prime[i, idx_top_i] * om_top_v[i] / G

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].scatter(fd_raw[mask_top], fd_C[mask_top], color=TOP_COLOR, s=50, label='Top-Heavy')
axes[0].scatter(fd_raw[mask_bot], fd_C[mask_bot], color=BOT_COLOR, s=50, marker='s', label='Bottom-Heavy')
axes[0].axhline(0, color='k', lw=0.5); axes[0].axvline(0, color='k', lw=0.5)
axes[0].set_xlabel(r'Method 2 $\langle\nabla\cdot(\mathbf{v}h)\rangle$ (W m$^{-2}$)')
axes[0].set_ylabel(r"Method 2 anomaly $\langle\nabla\cdot(\mathbf{v}h')\rangle$ (W m$^{-2}$)")
axes[0].set_title('Flux div: full h vs anomaly h\'')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].scatter(g1_vals[mask_top], gms_C[mask_top], color=TOP_COLOR, s=50, label='Top-Heavy')
axes[1].scatter(g1_vals[mask_bot], gms_C[mask_bot], color=BOT_COLOR, s=50, marker='s', label='Bottom-Heavy')
lim2 = 3
axes[1].plot([-lim2, lim2], [-lim2, lim2], 'k--', lw=0.8, label='1:1')
axes[1].set_xlim(-lim2, lim2); axes[1].set_ylim(-lim2, lim2)
axes[1].set_xlabel('GMS_adv (Method 1)')
axes[1].set_ylabel("GMS anomaly flux (Idea C)")
axes[1].set_title('GMS: Method 1 vs Idea C')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].hist(bnd_vals[mask_top],    bins=12, color=TOP_COLOR, alpha=0.5,  label='h original, Top')
axes[2].hist(bnd_anomaly[mask_top], bins=12, color=TOP_COLOR, alpha=0.9,  histtype='step', lw=2, label="h' anomaly, Top")
axes[2].hist(bnd_vals[mask_bot],    bins=12, color=BOT_COLOR, alpha=0.5,  label='h original, Bot')
axes[2].hist(bnd_anomaly[mask_bot], bins=12, color=BOT_COLOR, alpha=0.9,  histtype='step', lw=2, label="h' anomaly, Bot")
axes[2].axvline(0, color='k', lw=0.8)
axes[2].set_xlabel('Boundary term (W m$^{-2}$)')
axes[2].set_title("Boundary term: reduced with h'")
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

fig.suptitle("Idea C — Anomaly MSE h' = h − campaign mean", fontweight='bold')
plt.tight_layout()
plt.show()

for label, mask in [('Top-Heavy', mask_top), ('Bottom-Heavy', mask_bot)]:
    print(f"{label}: <∇·(vh)> = {np.nanmean(fd_raw[mask]):+8.1f} W/m²  →  anomaly = {np.nanmean(fd_C[mask]):+8.2f} W/m²"
          f"  |  boundary reduced: {np.nanmean(np.abs(bnd_vals[mask])):.1f} → {np.nanmean(np.abs(bnd_anomaly[mask])):.2f} W/m²")

## 8. GMS Comparison — All Approaches

In [ ]:
all_gms = {
    'GMS_adv\n(M1 raw)':          g1_vals,
    'GMS_flux\n(M2 raw)':         budget['gms_flux'].values,
    'Idea A\n(bnd correct.)':     gms_A,
    'Idea B\n(M1 mc only)':       gms_B,
    'Idea C\n(anomaly MSE)':      gms_C,
    'GMS_adv\n(M1 unified mc)':   g1_mc,
    'GMS_flux\n(M2 unified mc)':  g2_mc,
}

clip = 3
fig, axes = plt.subplots(1, 2, figsize=(17, 5), sharey=True)

for ax, mask, color, label in [
    (axes[0], mask_top, TOP_COLOR, 'Top-Heavy'),
    (axes[1], mask_bot, BOT_COLOR, 'Bottom-Heavy'),
]:
    data = [
        np.clip(v[mask][np.isfinite(v[mask])], -clip, clip)
        for v in all_gms.values()
    ]
    means = [np.nanmean(d) for d in data]
    pos   = np.arange(len(all_gms))

    bp = ax.boxplot(data, positions=pos, patch_artist=True,
                    widths=0.5, showfliers=False,
                    medianprops=dict(color='k', lw=2))
    for j, patch in enumerate(bp['boxes']):
        # shade the two unified-mc columns differently
        alpha = 0.8 if j >= 5 else 0.45
        patch.set_facecolor(color); patch.set_alpha(alpha)

    ax.plot(pos, means, 'k^', ms=8, zorder=5, label='group mean')
    ax.axhline(0, color='k', lw=0.8, ls='--')
    # vertical divider between raw/ideas and unified-mc columns
    ax.axvline(4.5, color='grey', lw=1.2, ls=':', label='unified mc →')
    ax.set_xticks(pos)
    ax.set_xticklabels(list(all_gms.keys()), fontsize=8.5)
    ax.set_ylabel('GMS')
    ax.set_title(f'{label} (n={mask.sum()})')
    ax.set_ylim(-clip, clip)
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(fontsize=9)

fig.suptitle('GMS across all methods and fix ideas  |  shaded = unified mass correction', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n{"Method":<32} {"Top-Heavy median":>18} {"Bottom-Heavy median":>20}')
print('-' * 72)
for mname, gms_vals in all_gms.items():
    name    = mname.replace('\n', ' ')
    gms_top = np.nanmedian(np.clip(gms_vals[mask_top], -10, 10))
    gms_bot = np.nanmedian(np.clip(gms_vals[mask_bot], -10, 10))
    print(f'{name:<32} {gms_top:>+18.4f} {gms_bot:>+20.4f}')

## Summary

### Problem
BEACH omega profiles end at ~16 km (below the tropopause), so ω_top ≠ 0.  
The open-boundary term `h_top·ω_top/g ≈ ±5000 W/m²` dominates Method 2/3 and makes them unusable without correction.

### Root cause of the fix
The fix (`omega_mass_corrected`) must zero omega at the **same** altitude level used by the column integrations (`_vadv_col`, `_col_int_p`). This requires h to be finite in the valid mask — without that, the ramp zeros omega at a higher unused level and the boundary term is unaffected.

### Results after unified mass correction (`mass_correct=True`)

| Quantity | Top-Heavy | Bottom-Heavy |
|---|---|---|
| Boundary term raw | ~−5519 W/m² | ~+307 W/m² |
| Boundary term mc | ~−17 W/m² | ~−14 W/m² |
| GMS_adv (group, raw) | +0.29 | −0.41 |
| GMS_adv (group, mc) | +0.32 | −0.34 |
| GMS_flux (group, mc) | +0.28 | +0.26 |
| Residual (div–ω) | ~22 W/m² | ~22 W/m² |

### Recommendation for RQ2

1. **Use `compute_budget(ds, mass_correct=True)`** as the standard analysis
2. **Report GMS as ratio of group means** — not mean of per-circle ratios
3. **Primary metric**: GMS_adv = `mean(vert_adv) / mean(vert_adv_dse)` per category
4. **Validation**: GMS_flux should agree within ~10% — if it does, report as confirmation
5. **Report** Δdiv mean as evidence that BEACH profiles are nearly mass-conserved (Δdiv ~ 10⁻⁷ s⁻¹ << typical div ~ 10⁻⁵ s⁻¹)

### Remaining limitation
The irreducible 22 W/m² residual is the div–ω inconsistency in BEACH (omega and div are independently derived from sonde data). This is ~10–20% of the signal — acceptable for the purposes of RQ2 but should be noted. ERA5 extension above the profile top would eliminate the open-boundary problem entirely.